In [1]:
import glob
import re
import pandas as pd
import shutil
from tqdm import tqdm

In [2]:
files = glob.glob("../data/K*")
f_name = []
for f in files:
    p = 'data/(K\d{6}).TXT'
    f_name.append(re.search(p,f).group(1))

In [3]:
len(f_name)

4

In [4]:
def mkcsv(FILE_NAME):
    with open('../data/' + FILE_NAME + '.TXT', encoding='shift-jis') as f:
        data = f.readlines()
    racers = []
    race = []
    place = []
    odds = []
    for i in data:
        if  re.match('^\s{3}第.+日', i):
            place.append(i.replace('\u3000','').replace('\n','').replace('/','_').replace('_ ', '_'))
        elif re.match('^\s+\d+R.+\s+H', i):
            race.append(i.replace('\u3000','').replace('\n',''))
        elif re.match('^\s{2}0[1-9]\s', i):
            racers.append(i.replace('\u3000','').replace('\n',''))
        elif re.match('^\s{5}.*[1-9\s][0-9]R\s{2}(\d-\d-\d|[^\s中\u3000止]+)', i):
            odds.append(i)

    pattern_place = re.compile('^.+(\d{4}_.+\d+)\s.+ボートレース([^0-9A-Z\s-]+)')
    pl = [re.match(pattern_place, i).groups() for i in place]

    pattern_race = re.compile('^\s+(\d+R)\s+.+\s{2}([^0-9A-Z\s-]+)\s{2}風\s{2}([^0-9A-Z\s]+).+(\d+)m.+波.+(\d+)cm')
    rc = [re.match(pattern_race, i).groups() for i in race]

    pattern_racers = re.compile('^\s{2}0([1-6])\s{2}([1-6])\s(\d{4})\s([^0-9]+)\s(\d+)\s+(\d+)\s{2}(\d{1}.\d{2})\s{3}(\d)\s{3}[\sF](\d.\d{2})\s+')
    rs = [re.match(pattern_racers, i).groups() for i in racers]

    pattern_odds = re.compile('^\s+(\d+R)\s{2}(\d-\d-\d|[^\s]*)\s*(\d+|\s)\s*(\d-\d-\d|[^\s]*)\s*(\d+|\s)\s+(\d-\d|[^\s]*)\s+(\d+|\s)\s+(\d-\d|[^\s]*)\s+(\d+|\s)')
    od = [re.match(pattern_odds, i).groups() for i in odds]

    column_rs = ['着','艇番','選手登番','選手名','モーター','ボート','展示','進入','スタートタイミング']
    df_rs = pd.DataFrame(rs, columns=column_rs)

    column_rc = ['R','天気','風向','風量','波']
    df_rc = pd.DataFrame(rc, columns=column_rc)

    column_pl = ['日時','場所']
    df_pl = pd.DataFrame(pl, columns=column_pl)
    df_rc['場所'] = ''
    df_rc['日時'] = ''

    column_od = ['R', '3連単_結果', '3連単_払戻', '3連複_結果', '3連複_払戻', '2連単_結果', '2連単_払戻', '2連複_結果', '2連複_払戻']
    df_od = pd.DataFrame(od, columns=column_od)
    for i in column_od:
        if i != "R":
            df_rc[i] = ''
    m = -1
    o = 0
    for i in range(df_rc.shape[0]):
        l = '1R'
        if i > 0:
            l = df_rc['R'][i-1]
        n = df_rc['R'][i]
        if n == '1R':
            m = m + 1
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]
            if l != n:
                o = o +1
            for j in column_od:
                if j == 'R':
                    pass
                else:
                    df_rc[j][i] = df_od[j][o]

        else:
            df_rc['場所'][i] = df_pl['場所'][m]
            df_rc['日時'][i] = df_pl['日時'][m]
            if l != n:
                o = o +1
            for j in column_od:
                if j == 'R':
                    pass
                else:
                    df_rc[j][i] = df_od[j][o]
    
    m = -1
    df_rs['場所'] = ''
    for c in df_rc.columns:
        df_rs[c] = ''

    for i in range(df_rs.shape[0]):
        n = int(df_rs['着'][i])
        if n == 1  :
            m = m + 1
            for c in df_rc.columns:
                df_rs[c][i] = df_rc[c][m]
        else:
            for c in df_rc.columns:
                df_rs[c][i] = df_rc[c][m]
    
    place_code = {'桐生':'01','戸田':'02','江戸川':'03','平和島':'04','多摩川':'05','浜名湖':'06','蒲郡':'07','常滑':'08','津':'09','三国':'10','びわこ':'11','住之江':'12','尼崎':'13','鳴門':'14','丸亀':'15','児島':'16','宮島':'17','徳山':'18','下関':'19','若松':'20','芦屋':'21','福岡':'22','唐津':'23','大村':'24'}

    df_rs['場所'] = df_rs['場所'].map(place_code)
    df_rs['RaceID'] = df_rs['日時'] + '_' + df_rs['場所'] + '_' + df_rs['R']

    df_rs.to_csv('../csv/' + FILE_NAME +  '.csv')
    shutil.move('../data/' + FILE_NAME + '.TXT', '../done/' + FILE_NAME + '.TXT')

In [6]:
for f in tqdm(f_name):
    #print(f)
    mkcsv(f)


100%|██████████| 4/4 [00:20<00:00,  5.06s/it]
